# ODE 3 notebook

## Shooting method

Consider a projectile moving in 2D. The equation of motion is:

$$
\frac{d^2\vec{r}}{dt^2} = \vec{a}
$$

with 

$$
\vec{a} = 0\hat{i} - 9.8\hat{j}
$$

To use the RK4 method, we split this second order equation into 2 first order equations:

$$
\frac{d\vec{r}}{dt} = \vec{v}
$$

and

$$
\frac{d\vec{r}}{dt} = \vec{a}
$$

If we know the initial position $(x_0, 0)$ and final position $(x_f, 0)$ but not the initial velocity, we have to use the shooting method to figure out the solution.

Here, we consider a projectile with $x_0 = 0$ m and $x_L = 30$ m. The ordering for solution $w$ is $(x, vx, y, vy)$. In this situation, there are a number of situations that would result in the correct distance. This is because the initial velocity depends on both the speed and angle at launch. If we had an additional boundary condition, such as a maximum height, we could find a unique solution. Here, we restrict the magnitude of the initial velocity to 25 m/s and adjust the angle.

In [22]:
import numpy as np 
import matplotlib.pyplot as plt 

# Constants
g = 9.81    # Gravitational constant
h = 0.001    # Step size in second for RK method
x0 = 0.
y0 = 0.
xL = 30.
yL = 30.
target = xL/10000.   # Target accuracy

# Define the functions for the physical system
def f(w, t):
    fx = w[1]
    fvx = 0
    fy = w[3]
    fvy = -g
    return np.array([fx, fvx, fy, fvy])

# Define a single step of the RK4 method
def rk4_step(w0, t0, h):
    k1 = h*f(w0, t0)
    k2 = h*f(w0+0.5*k1, t0+0.5*h)
    k3 = h*f(w0+0.5*k2, t0+0.5*h)
    k4 = h*f(w0+k3, t0+h)
    return w0 + (k1+2*k2+2*k3+k4)/6.0

# Calculate the position of the projectile when it returns to y = 0
def position(x0, y0, v0, h):
    vx0 = v0[0]
    vy0 = v0[1]

    time = np.array([0])
    w = np.array([[x0, vx0, y0, vy0]])

    # Take a first step 
    w = np.vstack([w, rk4_step(w[-1,:], time[-1], h)])
    time = np.append(time, time[-1]+h)

    # Past the first step, go until y <= 0.
    while w[-1,2] > 0:
        w = np.vstack([w, rk4_step(w[-1,:], time[-1], h)])
        time = np.append(time, time[-1]+h)

    return np.array([w[-1, 0], w[-1,2]])

# Binary search
## Find two vectors on each side of the solution.
theta_1 = np.pi/2
amp = 25
v1 = np.array([amp*np.cos(theta_1),amp*np.sin(theta_1)])     # Straight up in the air will definitely undershoot
d1 = position(x0, y0, v1, h)[0]
theta_2 = np.pi/4
v2 = np.array([amp*np.cos(theta_2), amp*np.sin(theta_2)])     # 45 degree projectile that defnitely overshoots
d2 = position(x0, y0, v2, h)[0]

while np.abs(d2-d1)>target:
    theta_p = (theta_1+theta_2)/2
    vp = np.array([amp*np.cos(theta_p), amp*np.sin(theta_p)])
    dp = position(x0, y0, vp, h)[0]

    if dp > xL:
        theta_2 = theta_p
        d2 = dp
    else:
        theta_1 = theta_p
        d1 = dp

theta = (theta_1+theta_2)/2
v = np.array([amp*np.cos(theta), amp*np.sin(theta)])
d = (d1+d2)/2
print(f"The initial angle is {theta*180/np.pi} degrees which results in an final position {position(x0, y0, v, h)}.")


The initial angle is 75.95603942871094 degrees which results in an final position [ 2.99996201e+01 -1.22605714e-02].
